# Depth Prediction Analysis

Interactive inspection of single test predictions on NYU Depth v2; it saves no paper
figure. The backbone is always **Depth Anything V2**; only the *predictive method*
(Det / Diag / LorD / SCORE / ...) varies, and it is inferred from the config that Hydra
logged alongside each run.

Given a logged run directory (`logs/.../<timestamp>/`, containing `.hydra/config.yaml`
and `checkpoints/`), `analyze_run(...)` rebuilds the model, loads the checkpoint,
runs one test sample, and produces:

1. **input · target · prediction · squared-error · predictive-std**
2. **target · three generated samples**

Unlike the ERA5 model, there is no residual reconstruction: the datamodule target
is the depth map directly (images and depths are stored scaled by `1/255`), so
every field is shown in that native scale.

## Setup

In [ ]:
import os, sys, math
from pathlib import Path
print(sys.executable, sys.version)

# --- Project-root bootstrap: put the repo root on sys.path so `import src...`
# works, and export PROJECT_ROOT for any Hydra ${oc.env:PROJECT_ROOT} lookups. ---
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".project-root").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("PROJECT_ROOT", str(PROJECT_ROOT))

import numpy as np
import torch
import hydra
from omegaconf import OmegaConf
from src.utils.legacy import upgrade_legacy_targets
import matplotlib.pyplot as plt

from src.methods.base import _as_list
from src.methods import PredictiveMethod, Deterministic
from src.models.gaussian_predictive import LightningPredictiveModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("project root:", PROJECT_ROOT, "| device:", DEVICE)

## Choose a run

Point `RUN_DIR` at any logged run directory. It must contain `.hydra/config.yaml`
and a `checkpoints/` folder. Both single-stage runs and SCORE runs (two-stage fit) work —
for the latter the stage-2 checkpoint is picked automatically.

## Dataloader from config

The datamodule is rebuilt from the *run's* config (not the repo's current
`configs/`), so its dataset and image size always match the checkpoint.
`preload` / `num_workers` are overridden for interactive use.

In [ ]:
_DM_CACHE = {}

def load_run_config(run_dir):
    """Load the fully-composed Hydra config logged with a run."""
    cfg_path = Path(run_dir) / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        raise FileNotFoundError(f"No logged config at {cfg_path}")
    return upgrade_legacy_targets(OmegaConf.load(cfg_path))

def build_datamodule(cfg, preload=False, num_workers=0, use_cache=True):
    """Instantiate the run's datamodule and set up the test split.

    `preload`/`num_workers` are overridden for interactive use (lazy per-sample
    reads, no worker processes). Cached by the data-shaping fields so repeated
    calls for the same config reuse one datamodule.
    """
    dm_cfg = cfg.datamodule
    key = (dm_cfg._target_, dm_cfg.get("data_path"), dm_cfg.get("dataset"), preload)
    if use_cache and key in _DM_CACHE:
        return _DM_CACHE[key]
    dm = hydra.utils.instantiate(dm_cfg, preload=preload, num_workers=num_workers)
    dm.setup("test")
    if use_cache:
        _DM_CACHE[key] = dm
    return dm

## Build the model and load the checkpoint

Mirrors `src.evaluation.evaluate`: the Depth Anything backbone and the predictive
method are rebuilt from config, then the checkpoint's weights are loaded into
them. SCORE runs (two-stage fit) use their stage-2 checkpoint.

In [ ]:
_MODEL_CACHE = {}

def find_checkpoint(run_dir):
    """Locate the deployed checkpoint in a logged run.

    Staged (whitened) runs deploy `checkpoints/stage2/*.ckpt`; plain runs keep a
    single `checkpoints/*.ckpt`. Returns the newest match.
    """
    ck = Path(run_dir) / "checkpoints"
    found = sorted(ck.glob("stage2/*.ckpt")) or sorted(ck.glob("*.ckpt"))
    if not found:
        raise FileNotFoundError(f"No .ckpt under {ck}")
    return str(found[-1])

def build_model_from_run(run_dir, device=DEVICE, datamodule=None,
                         preload=False, use_cache=True):
    """Rebuild the Depth Anything model for a logged run and load its checkpoint.

    Returns (model, predictive_method, datamodule, cfg, ckpt_path). The backbone
    is always `DepthAnything`; the predictive method is inferred from the config.
    """
    if use_cache and str(run_dir) in _MODEL_CACHE:
        return _MODEL_CACHE[str(run_dir)]
    cfg = load_run_config(run_dir)
    dm = datamodule or build_datamodule(cfg, preload=preload)

    # Eval bandwidth(s): gamma = sqrt(gamma_eval * d), d = target event dimension.
    d = dm.get_total_dimension()
    gamma_eval = [math.sqrt(f * d) for f in _as_list(cfg.get("gamma_eval", [1.0]))]

    method = hydra.utils.instantiate(
        cfg.method, out_channels=dm.get_channels(),
        gamma=gamma_eval, gamma_eval=gamma_eval)
    backbone = hydra.utils.instantiate(
        cfg.backbone, in_channels=dm.get_input_channels(),
        out_channels=method.out_channels, group_widths=method.group_widths)

    ckpt = find_checkpoint(run_dir)
    model = LightningPredictiveModel.load_from_checkpoint(
        ckpt, backbone=backbone, predictive_method=method, map_location=device)
    model.eval().to(device)
    method.training = False  # eval mode -> gamma_eval / n_eval_samples

    out = (model, method, dm, cfg, ckpt)
    if use_cache:
        _MODEL_CACHE[str(run_dir)] = out
    return out

## Analysis + plots

`analyze_run(...)` runs one test sample and returns a dict of *preliminary data*
(all fields plus metadata) and, by default, renders the two plots. Set
`make_plots=False` to get just the arrays.

The predictive standard deviation is the method's **analytic** per-pixel marginal
std — `method.get_marg_std(pred)`, the closed-form `sqrt(diag(Σ))` — so it is
exact for every method rather than a noisy ensemble estimate. Ensemble `samples`
are still drawn, but only for the generated-member plot. A deterministic run has
no spread, so its std is zero and its samples are degenerate.

In [ ]:
def _show(ax, data, title, cmap, vmin, vmax, cbar_label="depth"):
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    cb = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if cbar_label:
        cb.set_label(cbar_label)
    return im

def plot_prediction_summary(data):
    """input · target · prediction (mean) · squared error · predictive std."""
    target, mean = data["target"], data["mean"]
    vmin = float(min(target.min(), mean.min()))
    vmax = float(max(target.max(), mean.max()))
    u = data["units"]
    fig, axs = plt.subplots(1, 5, figsize=(25, 4.4))
    axs[0].imshow(np.clip(data["image"], 0, 1))
    axs[0].set_title("Input (RGB)"); axs[0].set_xticks([]); axs[0].set_yticks([])
    _show(axs[1], target, "Target", "Spectral_r", vmin, vmax, u)
    _show(axs[2], mean, "Prediction (mean)", "Spectral_r", vmin, vmax, u)
    _show(axs[3], data["mse_map"], "Squared error", "magma", 0, None, f"{u}²")
    _show(axs[4], data["std_map"], "Predictive std", "viridis", 0, None, u)
    fig.suptitle(f"{data['method']} · sample {data['sample_index']}", y=1.03)
    fig.tight_layout()
    return fig

def plot_samples(data, n_show=3):
    """target · n_show generated samples (shared color scale)."""
    target, mean, samples = data["target"], data["mean"], data["samples"]
    n_show = min(n_show, samples.shape[-1])
    show = samples[..., :n_show]
    vmin = float(min(target.min(), mean.min(), show.min()))
    vmax = float(max(target.max(), mean.max(), show.max()))
    u = data["units"]
    ncol = 1 + n_show
    fig, axs = plt.subplots(1, ncol, figsize=(5 * ncol, 4.4))
    _show(axs[0], target, "Target", "Spectral_r", vmin, vmax, u)
    for i in range(n_show):
        _show(axs[1 + i], show[..., i], f"Sample {i + 1}", "Spectral_r",
              vmin, vmax, u)
    fig.suptitle(f"{data['method']} · generated ensemble members", y=1.03)
    fig.tight_layout()
    return fig

In [ ]:
def analyze_run(run_dir, sample_index=0, n_samples=64, n_show_samples=3,
                device=DEVICE, datamodule=None, make_plots=True):
    """Build the model for a logged run, run one test sample, and summarize it.

    Returns a dict with the fields (`image`, `target`, `mean`, `samples` [H, W, M],
    `mse_map`, `std_map`), metadata, and — when `make_plots` is True — the two
    figures (`fig_summary`, `fig_samples`).

    The predictive `std_map` is the method's *analytic* per-pixel marginal std
    (`method.get_marg_std`, the closed-form sqrt(diag(Σ))), not an empirical
    ensemble estimate; `samples` are still drawn for the ensemble-member plot.
    """
    model, method, dm, cfg, ckpt = build_model_from_run(
        run_dir, device=device, datamodule=datamodule)

    x, y = dm.test_dataset[sample_index]
    x = x.unsqueeze(0).to(device)
    y = y.unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x)
        marg_std = method.get_marg_std(pred)  # (1, C, H, W); zeros for deterministic

    is_det = isinstance(method, Deterministic)
    mean = pred if is_det else pred[..., 0]
    samples = method.get_samples(pred, n_samples=n_samples)  # (1, C, H, W, M)

    image = x[0].detach().cpu().numpy().transpose(1, 2, 0)  # (H, W, 3), in [0, 1]
    target = y[0, 0].cpu().numpy()
    mean_map = mean[0, 0].cpu().numpy()
    samples_map = samples[0, 0].cpu().numpy()  # (H, W, M)

    mse_map = (mean_map - target) ** 2
    std_map = marg_std[0, 0].detach().cpu().numpy()  # (H, W)

    data = {
        "run_dir": str(run_dir),
        "method": cfg.method._target_.split(".")[-1],
        "loss": cfg.method.get("loss"),
        "ckpt": ckpt,
        "sample_index": sample_index,
        "units": "depth",
        "image": image,
        "target": target,
        "mean": mean_map,
        "samples": samples_map,
        "mse_map": mse_map,
        "std_map": std_map,
    }
    if is_det:
        print("Note: deterministic method -> predictive std and samples are degenerate.")
    if make_plots:
        data["fig_summary"] = plot_prediction_summary(data)
        data["fig_samples"] = plot_samples(data, n_show=n_show_samples)
    return data

# Example visualizations

In [ ]:
PRELOAD = False   # False = lazy per-sample reads (fast to open); True = whole split in RAM
sample_index = 0

cfg = load_run_config(PROJECT_ROOT / "logs/train/runs/2026-07-17_10-35-41")
datamodule = build_datamodule(cfg, preload=PRELOAD)
print("image size:", datamodule.test_dataset.get_image_size(),
      "| test samples:", len(datamodule.test_dataset))

# Det

In [ ]:
RUN_DIR = PROJECT_ROOT / "logs/train/runs/2026-07-17_10-35-41"

In [ ]:
result = analyze_run(RUN_DIR, sample_index=sample_index, n_samples=100, n_show_samples=3)

# The returned "preliminary data" (arrays + metadata); figures shown above.
print("method          :", result["method"], "| loss:", result["loss"])
print("checkpoint      :", result["ckpt"])
print("target shape    :", result["target"].shape,
      "| samples shape:", result["samples"].shape)
print("RMSE            : %.4f %s" % (np.sqrt(result["mse_map"].mean()), result["units"]))
print("mean pred. std  : %.4f %s" % (result["std_map"].mean(), result["units"]))
plt.show()

# LorD

In [ ]:
RUN_DIR = PROJECT_ROOT / "logs/train/runs/2026-07-17_13-07-47"

In [ ]:
result = analyze_run(RUN_DIR, sample_index=sample_index, n_samples=100, n_show_samples=3)

# The returned "preliminary data" (arrays + metadata); figures shown above.
print("method          :", result["method"], "| loss:", result["loss"])
print("checkpoint      :", result["ckpt"])
print("target shape    :", result["target"].shape,
      "| samples shape:", result["samples"].shape)
print("RMSE            : %.4f %s" % (np.sqrt(result["mse_map"].mean()), result["units"]))
print("mean pred. std  : %.4f %s" % (result["std_map"].mean(), result["units"]))
plt.show()

# SCORE

In [ ]:
RUN_DIR = PROJECT_ROOT / "logs/train/runs/2026-07-17_10-39-48" # 2 layer head

In [ ]:
RUN_DIR = PROJECT_ROOT / "logs/train/runs/2026-07-17_13-07-31" # 1 layer head

In [ ]:
result = analyze_run(RUN_DIR, sample_index=sample_index, n_samples=100, n_show_samples=3)

# The returned "preliminary data" (arrays + metadata); figures shown above.
print("method          :", result["method"], "| loss:", result["loss"])
print("checkpoint      :", result["ckpt"])
print("target shape    :", result["target"].shape,
      "| samples shape:", result["samples"].shape)
print("RMSE            : %.4f %s" % (np.sqrt(result["mse_map"].mean()), result["units"]))
print("mean pred. std  : %.4f %s" % (result["std_map"].mean(), result["units"]))
plt.show()